In [1]:
from google import genai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [8]:
complaint = input("Enter your complaint: ")

print("\nCustomer Complaint:")
print(complaint)

Enter your complaint:  My Sony headphones from order #12345 arrived damaged yesterday. I want a replacement.



Customer Complaint:
My Sony headphones from order #12345 arrived damaged yesterday. I want a replacement.


In [9]:
processing_prompt = f"""
You are an e-commerce customer support input processing system.

Analyze the following customer complaint and extract the important
information needed to resolve the complaint.

Customer Complaint:
{complaint}

Extract the following fields:

1. Order ID
2. Product
3. Issue
4. Customer Request
5. Date

Rules:
- Extract information only if it is present in the complaint.
- Do not invent or assume information.
- If information is not available, write "Not provided".
- Keep the extracted information concise.

Return the result exactly in this format:

Order ID: <value>
Product: <value>
Issue: <value>
Customer Request: <value>
Date: <value>
"""

In [10]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=processing_prompt
)

print(response.text)

Order ID: 12345
Product: Sony headphones
Issue: Arrived damaged
Customer Request: Replacement
Date: Yesterday


In [12]:
def process_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer support input processing system.

    Analyze the following customer complaint and extract the important
    information needed to resolve the complaint.

    Customer Complaint:
    {complaint}

    Extract:

    1. Order ID
    2. Product
    3. Issue
    4. Customer Request
    5. Date

    Rules:
    - Extract information only if it is present.
    - Do not invent or assume information.
    - If information is missing, write "Not provided".
    - Keep the information concise.

    Return exactly:

    Order ID: <value>
    Product: <value>
    Issue: <value>
    Customer Request: <value>
    Date: <value>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [13]:
complaint = input("Enter your complaint: ")

processed_input = process_complaint(complaint)

print("\nProcessed Information:")
print(processed_input)

Enter your complaint:  I ordered a black Nike shoe last week, but I received the wrong size. I want to exchange it.



Processed Information:
Order ID: Not provided
Product: Black Nike shoe
Issue: Received the wrong size
Customer Request: Exchange
Date: Last week


In [18]:
required_fields = [
    "Order ID",
    "Product",
    "Issue",
    "Customer Request",
    "Date"
]

In [19]:
def validate_processed_input(result):

    lines = result.splitlines()

    extracted_fields = {}

    for line in lines:
        for field in required_fields:
            if line.startswith(field + ":"):
                value = line.split(":", 1)[1].strip()
                extracted_fields[field] = value

    missing_fields = [
        field for field in required_fields
        if field not in extracted_fields
    ]

    return {
        "extracted_fields": extracted_fields,
        "all_fields_present": len(missing_fields) == 0,
        "missing_fields": missing_fields
    }

In [20]:
complaint = input("Enter your complaint: ")

processed_input = process_complaint(complaint)

validation = validate_processed_input(processed_input)

print("\nProcessed Information:")
print(processed_input)

print("\nValidation:")
print(validation)

Enter your complaint:  I ordered a black Nike shoe last week, but I received the wrong size. I want to exchange it.



Processed Information:
Order ID: Not provided
Product: Black Nike shoe
Issue: Received the wrong size
Customer Request: Exchange
Date: Last week

Validation:
{'extracted_fields': {'Order ID': 'Not provided', 'Product': 'Black Nike shoe', 'Issue': 'Received the wrong size', 'Customer Request': 'Exchange', 'Date': 'Last week'}, 'all_fields_present': True, 'missing_fields': []}
